# Interview Answer Scoring Model — Dataset Creation & Training

This cleaned notebook contains only the reproducible workflow used by the deployed application: loading the final dataset, generating NLP features, performing a group-aware train/test split, comparing regression models, validating with GroupKFold, training the final Random Forest model, saving it, and verifying the saved model.

## 1. Load the final 1000-row dataset

In [1]:
import pandas as pd

df = pd.read_csv("interview_dataset_1000.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (1000, 9)

Columns:
['seed_id', 'variation_id', 'question', 'expected_answer', 'candidate_answer', 'keywords', 'question_type', 'quality_label', 'quality_score']


In [2]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

c:\Users\Pradeep .k\OneDrive\Desktop\AI_Interview_Evaluator\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5303.84it/s]


Embedding model loaded successfully.


In [3]:
expected_embeddings = embedding_model.encode(
    df["expected_answer"].tolist(),
    show_progress_bar=True
)

candidate_embeddings = embedding_model.encode(
    df["candidate_answer"].tolist(),
    show_progress_bar=True
)

print("Expected embeddings shape:", expected_embeddings.shape)
print("Candidate embeddings shape:", candidate_embeddings.shape)

Batches: 100%|██████████| 32/32 [00:03<00:00,  8.32it/s]

Expected embeddings shape: (1000, 384)
Candidate embeddings shape: (1000, 384)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

df["semantic_score"] = [
    cosine_similarity(
        expected_embeddings[i].reshape(1, -1),
        candidate_embeddings[i].reshape(1, -1)
    )[0][0]
    for i in range(len(df))
]

print(df["semantic_score"].describe())

count    1000.000000
mean        0.836946
std         0.149375
min         0.341175
25%         0.780856
50%         0.877610
75%         0.952794
max         1.000000
Name: semantic_score, dtype: float64


In [5]:
import ast

df["keywords"] = df["keywords"].apply(ast.literal_eval)

print("Example keywords:")
print(df["keywords"].iloc[0])

print("\nNumber of keywords:",
      len(df["keywords"].iloc[0]))

Example keywords:
['python', 'high-level', 'programming language', 'syntax', 'readability']

Number of keywords: 5


In [6]:
def calculate_keyword_features(row):
    answer = row["candidate_answer"].lower()

    keywords = row["keywords"]

    matched = sum(
        1 for keyword in keywords
        if keyword.lower() in answer
    )

    total = len(keywords)

    score = matched / total if total > 0 else 0

    return pd.Series([
        matched,
        total,
        score
    ])


df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score"
    ]
] = df.apply(
    calculate_keyword_features,
    axis=1
)

print(df[
    [
        "matched_keywords",
        "total_keywords",
        "keyword_score"
    ]
].head(10))

   matched_keywords  total_keywords  keyword_score
0               4.0             5.0            0.8
1               4.0             5.0            0.8
2               4.0             5.0            0.8
3               4.0             5.0            0.8
4               4.0             5.0            0.8
5               4.0             5.0            0.8
6               4.0             5.0            0.8
7               4.0             5.0            0.8
8               4.0             5.0            0.8
9               4.0             5.0            0.8


In [7]:
df["answer_length"] = df["candidate_answer"].apply(
    lambda answer: len(answer.split())
)

df["expected_length"] = df["expected_answer"].apply(
    lambda answer: len(answer.split())
)

df["length_ratio"] = (
    df["answer_length"] /
    df["expected_length"]
)

print(
    df[
        [
            "answer_length",
            "expected_length",
            "length_ratio",
            "quality_label"
        ]
    ].head(10)
)

   answer_length  expected_length  length_ratio quality_label
0             28               13      2.153846     Excellent
1             30               13      2.307692     Excellent
2             31               13      2.384615     Excellent
3             32               13      2.461538     Excellent
4             29               13      2.230769     Excellent
5             21               13      1.615385          Good
6             23               13      1.769231          Good
7             24               13      1.846154          Good
8             25               13      1.923077          Good
9             22               13      1.692308          Good


In [8]:
import re

def calculate_sentence_features(answer):
    sentences = re.split(r'[.!?]+', answer)
    sentences = [s.strip() for s in sentences if s.strip()]

    sentence_count = len(sentences)

    if sentence_count > 0:
        avg_sentence_length = (
            len(answer.split()) / sentence_count
        )
    else:
        avg_sentence_length = 0

    return pd.Series([
        sentence_count,
        avg_sentence_length
    ])


df[
    [
        "sentence_count",
        "avg_sentence_length"
    ]
] = df["candidate_answer"].apply(
    calculate_sentence_features
)

print(
    df[
        [
            "answer_length",
            "sentence_count",
            "avg_sentence_length",
            "quality_label"
        ]
    ].head(10)
)

   answer_length  sentence_count  avg_sentence_length quality_label
0             28             2.0                 14.0     Excellent
1             30             2.0                 15.0     Excellent
2             31             2.0                 15.5     Excellent
3             32             2.0                 16.0     Excellent
4             29             2.0                 14.5     Excellent
5             21             2.0                 10.5          Good
6             23             2.0                 11.5          Good
7             24             2.0                 12.0          Good
8             25             2.0                 12.5          Good
9             22             2.0                 11.0          Good


In [9]:
def calculate_unique_word_ratio(answer):
    words = answer.lower().split()

    if len(words) == 0:
        return 0

    unique_words = set(words)

    return len(unique_words) / len(words)


df["unique_word_ratio"] = df["candidate_answer"].apply(
    calculate_unique_word_ratio
)

print(
    df[
        [
            "answer_length",
            "unique_word_ratio",
            "quality_label"
        ]
    ].head(10)
)

   answer_length  unique_word_ratio quality_label
0             28           0.928571     Excellent
1             30           0.900000     Excellent
2             31           0.870968     Excellent
3             32           0.906250     Excellent
4             29           0.931034     Excellent
5             21           0.904762          Good
6             23           0.869565          Good
7             24           0.833333          Good
8             25           0.880000          Good
9             22           0.909091          Good


In [10]:
feature_columns = [
    "semantic_score",
    "keyword_score",
    "answer_length",
    "length_ratio",
    "matched_keywords",
    "sentence_count",
    "avg_sentence_length",
    "unique_word_ratio"
]

X = df[feature_columns]
y = df["quality_score"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(feature_columns)

Feature matrix shape: (1000, 8)
Target shape: (1000,)

Features:
['semantic_score', 'keyword_score', 'answer_length', 'length_ratio', 'matched_keywords', 'sentence_count', 'avg_sentence_length', 'unique_word_ratio']


In [11]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["seed_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 800
Testing samples: 200


In [12]:
train_seeds = set(df.iloc[train_idx]["seed_id"])
test_seeds = set(df.iloc[test_idx]["seed_id"])

overlap = train_seeds.intersection(test_seeds)

print("Training seeds:", len(train_seeds))
print("Testing seeds:", len(test_seeds))
print("Overlapping seeds:", len(overlap))
print("Overlap:", overlap)

Training seeds: 160
Testing seeds: 40
Overlapping seeds: 0
Overlap: set()


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42
    )
}

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="MAE"
).reset_index(drop=True)

results_df

,Model,MAE,R2
0,Decision Tree,4.500000,0.851240
1,Random Forest,4.624375,0.925150
2,Extra Trees,4.831875,0.913434
3,Gradient Boosting,5.695476,0.912373
4,Linear Regression,7.756846,0.881705


In [14]:
from sklearn.model_selection import GroupKFold, cross_validate

cv = GroupKFold(n_splits=5)

cv_results = []

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        groups=df.iloc[train_idx]["seed_id"],
        scoring={
            "MAE": "neg_mean_absolute_error",
            "R2": "r2"
        },
        n_jobs=-1
    )

    mean_mae = -scores["test_MAE"].mean()
    mean_r2 = scores["test_R2"].mean()

    cv_results.append({
        "Model": name,
        "Mean MAE": mean_mae,
        "Mean R2": mean_r2
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by="Mean MAE"
).reset_index(drop=True)

cv_results_df

,Model,Mean MAE,Mean R2
0,Decision Tree,6.000000,0.803361
1,Random Forest,6.270469,0.871129
2,Extra Trees,6.459844,0.859270
3,Gradient Boosting,6.757562,0.875282
4,Linear Regression,8.397769,0.857352


In [15]:
final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

final_model.fit(
    X,
    y
)

print("Final deployment model trained.")
print("Training samples used:", len(X))

Final deployment model trained.
Training samples used: 1000


In [16]:
import joblib
import os

joblib.dump(
    final_model,
    "interview_score_model.pkl"
)

print("Final model saved successfully.")
print(
    "File exists:",
    os.path.exists("interview_score_model.pkl")
)

Final model saved successfully.
File exists: True


In [17]:
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": final_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance

,Feature,Importance
0,answer_length,0.737836
1,length_ratio,0.097419
2,semantic_score,0.082794
3,avg_sentence_length,0.040260
4,keyword_score,0.020069
5,unique_word_ratio,0.011814
6,matched_keywords,0.009718
7,sentence_count,0.000089


In [18]:
for i, question in enumerate(sorted(df["question"].unique()), start=1):
    print(f"{i}. {question}")

1. What is NumPy?
2. What is Pandas?
3. What is Python?
4. What is SQL?
5. What is a DataFrame in Pandas?
6. What is a JOIN in SQL?
7. What is a Python function?
8. What is a SQL GROUP BY clause?
9. What is a SQL query?
10. What is a confusion matrix?
11. What is a dictionary in Python?
12. What is a foreign key in SQL?
13. What is a histogram?
14. What is a list in Python?
15. What is a machine learning model?
16. What is a primary key in SQL?
17. What is a queue in data structures?
18. What is a stack in data structures?
19. What is a tuple in Python?
20. What is abstraction in OOP?
21. What is accuracy in machine learning?
22. What is an INNER JOIN in SQL?
23. What is an outlier?
24. What is binary search?
25. What is classification in machine learning?
26. What is cosine similarity?
27. What is cross-validation?
28. What is data preprocessing?
29. What is database normalization?
30. What is encapsulation in OOP?
31. What is exploratory data analysis?
32. What is feature engineering

## Final model

The deployment model is the Random Forest regressor trained on the complete feature matrix `X` and target `y`. The saved file `interview_score_model.pkl` is loaded by the Flask application.